# **Name: Manahil Shahid**
# **Reg No: SP24-BDS-023**
# **Class: BDS-6A**
# **Subject: Natural Language Processing**
# **Lab 05: Lab Tasks**

## Setup

In this lab I build small N gram language models from scratch. I only use the Python standard library: math for logarithms, itertools for permutations and collections.Counter for counting. Below I write the training and test data. In every task I lowercase everything, add <s> and </s>, use the natural log (ln) and round to 4 decimals.

In [1]:
import math
import itertools
from collections import Counter

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test  = ["the cat sat on the rug", "the fish sat on the mat"]

print("Training sentences:", len(train))
print("Test sentences:", len(test))

Training sentences: 4
Test sentences: 2


#### What the output shows

The output confirms that I have 4 training sentences and 2 test sentences. These two small lists are used in every task below.

## Task 1: Tokenization

In this task I turn each sentence into a list of tokens, count the tokens and build the vocabulary. I am solving it in four parts.

### Part 1 and 2: Write tokenize and apply it

Here I write tokenize(sentence). It changes the sentence to lowercase, splits it on spaces and puts <s> at the start and </s> at the end. These two markers tell the model where a sentence begins and where it ends. Then I apply the function to every training and test sentence and print the results.

In [2]:
def tokenize(sentence):
    words = sentence.lower().split(" ")
    return ["<s>"] + words + ["</s>"]

train_tokens = [tokenize(s) for s in train]
test_tokens = [tokenize(s) for s in test]

print("Training sentences")
for tokens in train_tokens:
    print(tokens)

print()
print("Test sentences")
for tokens in test_tokens:
    print(tokens)

Training sentences
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']

Test sentences
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']


#### What the output shows

Every sentence is now a list of words with <s> in front and </s> at the back. The four training sentences have 8, 7, 8 and 7 tokens, and the two test sentences have 8 tokens each. The word the appears twice in most sentences, and it stays twice in the list.

### Part 3: Count the tokens in the training corpus

Here I count all the tokens in the four training lists. I count once with the markers and once without them. Without markers I just remove 2 from the length of each list, because every list has exactly one <s> and one </s>.

In [3]:
with_markers = 0
without_markers = 0
for tokens in train_tokens:
    with_markers += len(tokens)
    without_markers += len(tokens) - 2

print("Total tokens with markers   :", with_markers)
print("Total tokens without markers:", without_markers)

Total tokens with markers   : 30
Total tokens without markers: 22


#### What the output shows

The corpus has 30 tokens with markers and 22 tokens without markers. The difference is 8, which is 4 sentences times 2 markers.

### Part 4: Build the vocabulary

Here I write build_vocab(sentences). It collects every different word in a set, adds </s> and returns the sorted list. I do not add <s> because the model never has to predict the start of a sentence, it only has to predict what comes after it. So V is the number of different words plus </s>.

In [4]:
def build_vocab(sentences):
    vocab = set()
    for sentence in sentences:
        for word in sentence.lower().split(" "):
            vocab.add(word)
    vocab.add("</s>")
    return sorted(vocab)

vocab = build_vocab(train)
V = len(vocab)

print("V =", V)
print("Sorted vocabulary:", vocab)

V = 11
Sorted vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


#### What the output shows

V is 11. That is the 10 different words of the corpus plus </s>. In the sorted list </s> comes first because the symbol < is sorted before the letters. I use this V later in smoothing.

## Task 2: Generating N grams

An N gram is a group of N tokens that come one after the other. In this task I write a function that makes them and I check that it works correctly.

### Part 1 and 2: Write get_ngrams and print the N grams of the first sentence

Here I write get_ngrams(tokens, n). It slides a window of size n over the token list, one step at a time, and saves every window as a tuple. Then I print the unigrams (n = 1), bigrams (n = 2) and trigrams (n = 3) of the first training sentence.

In [5]:
def get_ngrams(tokens, n):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i:i + n]))
    return ngrams

first = train_tokens[0]
print("Tokens   :", first)
print()
print("Unigrams :", get_ngrams(first, 1))
print()
print("Bigrams  :", get_ngrams(first, 2))
print()
print("Trigrams :", get_ngrams(first, 3))

Tokens   : ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']

Unigrams : [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]

Bigrams  : [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]

Trigrams : [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]


#### What the output shows

The first sentence has 8 tokens. It gives 8 unigrams, 7 bigrams and 6 trigrams. Each bigram shares one word with the next bigram, and each trigram shares two words with the next trigram, because the window moves only one step at a time.

### Part 3: Verify the number of N grams

Here I check a rule with code. A sentence with m tokens must give m minus n plus 1 N grams. I loop over all 6 sentences (4 training and 2 test) and for n = 1, 2 and 3 I use assert to compare the real number with the expected number. If the rule was wrong, assert would stop the program with an error.

In [6]:
all_tokens = train_tokens + test_tokens

for tokens in all_tokens:
    m = len(tokens)
    counts = []
    for n in [1, 2, 3]:
        found = len(get_ngrams(tokens, n))
        expected = m - n + 1
        assert found == expected
        counts.append(found)
    print("m =", m, "| n=1, 2, 3 give", counts, "| expected", [m, m - 1, m - 2])

print()
print("All checks passed")

m = 8 | n=1, 2, 3 give [8, 7, 6] | expected [8, 7, 6]
m = 7 | n=1, 2, 3 give [7, 6, 5] | expected [7, 6, 5]
m = 8 | n=1, 2, 3 give [8, 7, 6] | expected [8, 7, 6]
m = 7 | n=1, 2, 3 give [7, 6, 5] | expected [7, 6, 5]
m = 8 | n=1, 2, 3 give [8, 7, 6] | expected [8, 7, 6]
m = 8 | n=1, 2, 3 give [8, 7, 6] | expected [8, 7, 6]

All checks passed


#### What the output shows

For a sentence with m = 8 the function gave 8, 7 and 6 N grams, and for m = 7 it gave 7, 6 and 5. These match the expected numbers on every line, and the message All checks passed appears at the end, so the rule is verified.

### Part 4: What happens when n is bigger than the sentence

Here I test a very short sentence with only 3 tokens and ask for 5 grams and 4 grams. In my function the range becomes empty when n is bigger than the length, so the loop does not run and the function returns an empty list instead of crashing.

In [7]:
short = ["<s>", "hi", "</s>"]
print("Sentence has", len(short), "tokens")
print("n = 5 gives:", get_ngrams(short, 5))
print("n = 4 gives:", get_ngrams(short, 4))
print("n = 3 gives:", get_ngrams(short, 3))

Sentence has 3 tokens
n = 5 gives: []
n = 4 gives: []
n = 3 gives: [('<s>', 'hi', '</s>')]


#### What the output shows

With n = 5 and n = 4 the function returns an empty list, and there is no error. With n = 3 it returns exactly one trigram, which is the whole sentence.

## Task 3: Counting N grams

In this task I count how many times each N gram appears in the training corpus. These counts are the base of all the probabilities that come later.

### Part 1: Build the counters

Here I make three Counter objects, one for unigrams, one for bigrams and one for trigrams. For every training sentence I give the tokens to the counter with update(), and it adds 1 for each N gram it sees. The unigram counter stores plain words, and the bigram and trigram counters store tuples.

In [8]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update(tokens)
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))

print("Counters built")

Counters built


#### What the output shows

The output only says that the counters were built. The numbers are printed in the next part.

### Part 2: Print the count tables

Here I print each table from the highest count to the lowest count. When two N grams have the same count, they are sorted alphabetically so the order is always the same.

In [9]:
def print_counts(counter, title):
    print(title)
    items = sorted(counter.items(), key=lambda x: (-x[1], x[0]))
    for gram, c in items:
        if isinstance(gram, str):
            name = gram
        else:
            name = " ".join(gram)
        print(f"  {name:<22} {c}")
    print()

print_counts(unigram_counts, "Unigram counts")
print_counts(bigram_counts, "Bigram counts")
print_counts(trigram_counts, "Trigram counts")

Unigram counts
  the                    8
  </s>                   4
  <s>                    4
  ate                    2
  cat                    2
  dog                    2
  on                     2
  sat                    2
  bone                   1
  fish                   1
  mat                    1
  rug                    1

Bigram counts
  <s> the                4
  ate the                2
  on the                 2
  sat on                 2
  the cat                2
  the dog                2
  bone </s>              1
  cat ate                1
  cat sat                1
  dog ate                1
  dog sat                1
  fish </s>              1
  mat </s>               1
  rug </s>               1
  the bone               1
  the fish               1
  the mat                1
  the rug                1

Trigram counts
  <s> the cat            2
  <s> the dog            2
  sat on the             2
  ate the bone           1
  ate the fish           1
  cat ate

#### What the output shows

The word the has the highest unigram count, 8, because it appears twice in each sentence. The markers <s> and </s> each have a count of 4 because there is one of each in every sentence. The most common bigram is <s> the with count 4, because all four sentences start with the. In the trigram table the top counts are 2 for <s> the cat, <s> the dog and sat on the, and every other trigram appears only once.

### Part 3: Most common bigrams and number of distinct N grams

Here I use most_common(5) to print the 5 bigrams with the highest counts. Then I print how many different unigrams, bigrams and trigrams there are, using the length of each counter.

In [10]:
print("5 most common bigrams")
for gram, c in bigram_counts.most_common(5):
    print(" ", " ".join(gram), c)

print()
print("Distinct unigrams:", len(unigram_counts))
print("Distinct bigrams :", len(bigram_counts))
print("Distinct trigrams:", len(trigram_counts))

5 most common bigrams
  <s> the 4
  the cat 2
  sat on 2
  on the 2
  ate the 2

Distinct unigrams: 12
Distinct bigrams : 18
Distinct trigrams: 19


#### What the output shows

The top bigram is <s> the with 4, followed by bigrams with count 2. The bigram the dog also has count 2, but most_common(5) shows only five. The corpus has 12 different unigrams (this includes both markers), 18 different bigrams and 19 different trigrams.

### Part 4: Write count_of

Here I write count_of(ngram). It looks at the length of the N gram and uses the matching counter. I use .get(ngram, 0) so that an N gram that was never seen gives 0 and does not raise a KeyError. I test it on one seen bigram and one unseen bigram.

In [11]:
def count_of(ngram):
    if len(ngram) == 1:
        return unigram_counts.get(ngram[0], 0)
    if len(ngram) == 2:
        return bigram_counts.get(ngram, 0)
    return trigram_counts.get(ngram, 0)

print("Seen bigram   ('the', 'cat') :", count_of(("the", "cat")))
print("Unseen bigram ('fish', 'sat'):", count_of(("fish", "sat")))

Seen bigram   ('the', 'cat') : 2
Unseen bigram ('fish', 'sat'): 0


#### What the output shows

The seen bigram (the, cat) gives 2, which matches the bigram table. The unseen bigram (fish, sat) gives 0 and the program does not crash.

## Task 4: Unigram model

A unigram model gives each word a probability that does not depend on the words around it. In this task I build it with maximum likelihood estimation (MLE) and test it on the two test sentences.

### Part 1 and 2: Unigram probabilities

Here I write unigram_prob(w) with the formula P(w) = C(w) / N. N is the total number of tokens the model can predict. I do not count <s> in N because the model never predicts it, so N is 22 words plus 4 copies of </s>, which is 26. Then I print all the probabilities in a table and use assert to check that they add up to 1.

In [12]:
N = sum(unigram_counts.values()) - unigram_counts["<s>"]

def unigram_prob(w):
    return unigram_counts.get(w, 0) / N

print("N =", N)
print()
print(f"{'word':<8} {'count':<6} {'P(w)'}")
total = 0
for w in vocab:
    p = unigram_prob(w)
    total += p
    print(f"{w:<8} {unigram_counts[w]:<6} {round(p, 4)}")

print()
print("Sum of probabilities:", round(total, 4))
assert abs(total - 1) < 1e-9

N = 26

word     count  P(w)
</s>     4      0.1538
ate      2      0.0769
bone     1      0.0385
cat      2      0.0769
dog      2      0.0769
fish     1      0.0385
mat      1      0.0385
on       2      0.0769
rug      1      0.0385
sat      2      0.0769
the      8      0.3077

Sum of probabilities: 1.0


#### What the output shows

N is 26. The word the gets 8 / 26, which is 0.3077, and the rare words like bone, fish, mat and rug get 1 / 26, which is 0.0385. The sum of all probabilities is 1.0 and the assert passed, so the table is a proper probability distribution.

### Part 3: Probability of a sentence

Here I write unigram_sentence_prob(sentence). It multiplies the probability of every token after <s>, including </s>. The numbers are very small, so I print the probability and also its ln value rounded to 4 decimals.

In [13]:
def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = 1
    for w in tokens[1:]:
        p = p * unigram_prob(w)
    return p

for s in test:
    p = unigram_sentence_prob(s)
    print(s)
    print("  probability:", f"{p:.6e}", "| ln:", round(math.log(p), 4))

the cat sat on the rug
  probability: 2.549861e-07 | ln: -15.1821
the fish sat on the mat
  probability: 1.274931e-07 | ln: -15.8752


#### What the output shows

The first test sentence has probability about 0.00000025 and ln about negative 15.1821. The second has probability about 0.00000013 and ln about negative 15.8752. The first one is exactly twice as likely, because it has cat (count 2) where the second has fish (count 1), and all other words are the same.

### Part 4: Word order does not matter

Here I take the words of the first test sentence and use itertools.permutations to make every possible word order. I use a set to remove repeated orders, because the appears twice. Then I find the probability of every order and collect the ln values in another set. If the set has only one value, all orders have the same probability.

In [14]:
words = test[0].split(" ")
all_orders = set(itertools.permutations(words))
print("Words:", words)
print("Different word orders:", len(all_orders))

ln_values = set()
for order in all_orders:
    p = unigram_sentence_prob(" ".join(order))
    ln_values.add(round(math.log(p), 4))

print("Different ln probability values:", len(ln_values), ln_values)
print()
print("Three example orders")
for order in sorted(all_orders)[:3]:
    sentence = " ".join(order)
    print(" ", sentence, "| ln:", round(math.log(unigram_sentence_prob(sentence)), 4))

Words: ['the', 'cat', 'sat', 'on', 'the', 'rug']
Different word orders: 360
Different ln probability values: 1 {-15.1821}

Three example orders
  cat on rug sat the the | ln: -15.1821
  cat on rug the sat the | ln: -15.1821
  cat on rug the the sat | ln: -15.1821


#### What the output shows

There are 360 different word orders, and all of them give the same ln value, negative 15.1821, so the set has only one value. The three examples show nonsense sentences like cat on rug sat the the that get exactly the same score as the real sentence. This happens because multiplication does not care about order. It is the main weakness of a unigram model.

## Task 5: Bigram model (MLE)

A bigram model predicts a word using only the one word before it. In this task I build it with MLE and use it to find sentence probabilities and to predict the next word.

### Part 1: Bigram probability

Here I write bigram_prob(prev, w) with the formula P(w | v) = C(v, w) / C(v). If the previous word was never seen, the function returns 0 so that there is no division by zero.

In [15]:
def bigram_prob(prev, w):
    c_prev = unigram_counts.get(prev, 0)
    if c_prev == 0:
        return 0
    return bigram_counts.get((prev, w), 0) / c_prev

print("P(cat | the) =", bigram_prob("the", "cat"))
print("P(sat | fish) =", bigram_prob("fish", "sat"))

P(cat | the) = 0.25
P(sat | fish) = 0.0


#### What the output shows

P(cat | the) is 2 / 8, which is 0.25, because the appears 8 times and 2 of them are followed by cat. P(sat | fish) is 0.0 because fish is never followed by sat in the training data.

### Part 2: Next word distribution

Here I write next_word_distribution(prev). It goes through the vocabulary and saves every word that has a probability bigger than 0 in a dictionary. I print it for the and use assert to check that the probabilities sum to 1.

In [16]:
def next_word_distribution(prev):
    dist = {}
    for w in vocab:
        p = bigram_prob(prev, w)
        if p > 0:
            dist[w] = p
    return dist

dist_the = next_word_distribution("the")
for w, p in sorted(dist_the.items(), key=lambda x: (-x[1], x[0])):
    print(f"  P({w} | the) = {round(p, 4)}")

total = sum(dist_the.values())
print("Sum:", round(total, 4))
assert abs(total - 1) < 1e-9

  P(cat | the) = 0.25
  P(dog | the) = 0.25
  P(bone | the) = 0.125
  P(fish | the) = 0.125
  P(mat | the) = 0.125
  P(rug | the) = 0.125
Sum: 1.0


#### What the output shows

After the, six different words can come. Cat and dog each have 0.25, and bone, fish, mat and rug each have 0.125. The sum is 1.0 and the assert passed.

### Part 3: Probability of a sentence with the chain rule

Here I write bigram_sentence_prob(sentence). By the chain rule I multiply the bigram probability of each token given the token before it, starting from <s> and ending with </s>.

In [17]:
def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = 1
    for i in range(1, len(tokens)):
        p = p * bigram_prob(tokens[i - 1], tokens[i])
    return p

for s in test:
    p = bigram_sentence_prob(s)
    print(s)
    if p > 0:
        print("  probability:", f"{p:.6e}", "| ln:", round(math.log(p), 4))
    else:
        print("  probability: 0 | ln: negative infinity")

the cat sat on the rug
  probability: 1.562500e-02 | ln: -4.1589
the fish sat on the mat
  probability: 0 | ln: negative infinity


#### What the output shows

The first test sentence has probability 0.015625 (this is 1 / 64) and ln about negative 4.1589. This is much higher than the unigram result, because the bigram model uses the previous word. The second test sentence has probability 0. All of its factors are fine except one, P(sat | fish), which is 0, and one zero makes the whole product zero.

### Part 4: Predict the next word

Here I write predict_next(prev). It takes the next word distribution and returns the word with the highest probability. When two words have the same probability, the alphabetical order decides, so the answer is always the same. I test it on 8 different words.

In [18]:
def predict_next(prev):
    dist = next_word_distribution(prev)
    if len(dist) == 0:
        return None
    ranked = sorted(dist.items(), key=lambda x: (-x[1], x[0]))
    return ranked[0][0]

for w in ["<s>", "the", "cat", "dog", "sat", "on", "ate", "fish"]:
    print(f"After {w:<4} the most probable next word is: {predict_next(w)}")

After <s>  the most probable next word is: the
After the  the most probable next word is: cat
After cat  the most probable next word is: ate
After dog  the most probable next word is: ate
After sat  the most probable next word is: on
After on   the most probable next word is: the
After ate  the most probable next word is: the
After fish the most probable next word is: </s>


#### What the output shows

After <s> the model predicts the, after sat it predicts on, after on and ate it predicts the, and after fish it predicts </s>. After the, cat and dog are tied at 0.25 and cat wins because it comes first alphabetically. After cat and after dog, ate and sat are tied and ate wins in the same way.

## Task 6: Trigram model

A trigram model predicts a word using the two words before it. In this task I build it and compare how many N grams of the test sentences were never seen.

### Part 1: Trigram probability

Here I write trigram_prob(w1, w2, w3) with the formula P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2). If the pair w1, w2 was never seen, the function returns 0.

In [19]:
def trigram_prob(w1, w2, w3):
    c_pair = bigram_counts.get((w1, w2), 0)
    if c_pair == 0:
        return 0
    return trigram_counts.get((w1, w2, w3), 0) / c_pair

print("P(sat | the, cat) =", trigram_prob("the", "cat", "sat"))
print("P(ate | the, cat) =", trigram_prob("the", "cat", "ate"))
print("P(rug | on, the)  =", trigram_prob("on", "the", "rug"))

P(sat | the, cat) = 0.5
P(ate | the, cat) = 0.5
P(rug | on, the)  = 0.5


#### What the output shows

The pair the cat appears 2 times. One time it is followed by sat and one time by ate, so both probabilities are 0.5. The pair on the appears 2 times and is followed by rug once, so P(rug | on, the) is also 0.5.

### Part 2: Probability of a sentence

Here I write trigram_sentence_prob(sentence). The first word has only <s> before it, so for the first word I use the bigram model P(word | <s>). For all the other words I use the trigram model with the two words before.

In [20]:
def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = bigram_prob("<s>", tokens[1])
    for i in range(2, len(tokens)):
        p = p * trigram_prob(tokens[i - 2], tokens[i - 1], tokens[i])
    return p

for s in test:
    p = trigram_sentence_prob(s)
    print(s)
    if p > 0:
        print("  probability:", f"{p:.6e}", "| ln:", round(math.log(p), 4))
    else:
        print("  probability: 0 | ln: negative infinity")

the cat sat on the rug
  probability: 1.250000e-01 | ln: -2.0794
the fish sat on the mat
  probability: 0 | ln: negative infinity


#### What the output shows

The first test sentence has probability 0.125 and ln about negative 2.0794. This is higher than the bigram result of 0.015625, because the two words of context make the model more sure about each choice. The second test sentence is again 0, because the trigram the fish sat was never seen.

### Part 3: Table of distinct N grams

Here I print the number of different unigrams, bigrams and trigrams in a small table.

In [21]:
print(f"{'N-gram type':<12} {'Distinct count'}")
print(f"{'Unigrams':<12} {len(unigram_counts)}")
print(f"{'Bigrams':<12} {len(bigram_counts)}")
print(f"{'Trigrams':<12} {len(trigram_counts)}")

N-gram type  Distinct count
Unigrams     12
Bigrams      18
Trigrams     19


#### What the output shows

There are 12 different unigrams, 18 different bigrams and 19 different trigrams. The number grows when N grows, and since each one is more specific, each one is seen fewer times.

### Part 4: Unseen N grams in the test sentences

Here I list, for each test sentence, the bigrams and trigrams that have a training count of 0. Then I compare the two numbers to see which model meets more unseen N grams.

In [22]:
for s, tokens in zip(test, test_tokens):
    unseen_bi = [g for g in get_ngrams(tokens, 2) if count_of(g) == 0]
    unseen_tri = [g for g in get_ngrams(tokens, 3) if count_of(g) == 0]
    print(s)
    print("  Unseen bigrams :", unseen_bi, "->", len(unseen_bi))
    print("  Unseen trigrams:", unseen_tri, "->", len(unseen_tri))
    if len(unseen_tri) > len(unseen_bi):
        print("  The trigram model has more unseen N-grams")
    elif len(unseen_bi) > len(unseen_tri):
        print("  The bigram model has more unseen N-grams")
    else:
        print("  Both models have the same number of unseen N-grams")
    print()

the cat sat on the rug
  Unseen bigrams : [] -> 0
  Unseen trigrams: [] -> 0
  Both models have the same number of unseen N-grams

the fish sat on the mat
  Unseen bigrams : [('fish', 'sat')] -> 1
  Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')] -> 3
  The trigram model has more unseen N-grams



#### What the output shows

The first test sentence has no unseen bigram and no unseen trigram. The second test sentence has 1 unseen bigram, (fish, sat), and 3 unseen trigrams, which are (<s>, the, fish), (the, fish, sat) and (fish, sat, on). So the trigram model has more unseen N grams. One unseen word pair spoils every trigram that contains it, so longer N grams run into unseen cases more often.

## Task 7: Zero probability problem

In this task I look at what goes wrong when a test sentence has a bigram that never appeared in training.

### Part 1 and 2: Find the unseen bigrams

Here I write unseen_bigrams(sentence). It returns every bigram of the sentence that has a training count of 0. I run it on both test sentences and print the bigram MLE probability of each. Then I use assert to confirm that one of the probabilities is 0.

In [23]:
def unseen_bigrams(sentence):
    tokens = tokenize(sentence)
    missing = []
    for g in get_ngrams(tokens, 2):
        if bigram_counts.get(g, 0) == 0:
            missing.append(g)
    return missing

mle_values = []
for s in test:
    p = bigram_sentence_prob(s)
    mle_values.append(p)
    print(s)
    print("  Unseen bigrams:", unseen_bigrams(s))
    print("  Bigram MLE probability:", p)

assert 0 in mle_values
print()
print("Confirmed: at least one test sentence has probability 0")

the cat sat on the rug
  Unseen bigrams: []
  Bigram MLE probability: 0.015625
the fish sat on the mat
  Unseen bigrams: [('fish', 'sat')]
  Bigram MLE probability: 0.0

Confirmed: at least one test sentence has probability 0


#### What the output shows

The first sentence has no unseen bigram and its probability is 0.015625. The second sentence has the unseen bigram (fish, sat) and its probability is 0.0. The assert passed, so I confirmed that one sentence gets a zero probability, even though it is a normal English sentence.

### Part 3: How many word pairs never occur

Here I count the word pairs that never occur in training. I use only real words, so the markers are left out. There are 10 words, so there are 10 times 10 = 100 possible ordered pairs. I count the bigrams in the training counter that have no marker, subtract them from 100 and write the result as a percentage.

In [24]:
word_list = [w for w in vocab if w != "</s>"]
total_pairs = len(word_list) * len(word_list)

seen_pairs = 0
for (a, b) in bigram_counts:
    if a not in ["<s>", "</s>"] and b not in ["<s>", "</s>"]:
        seen_pairs += 1

unseen_pairs = total_pairs - seen_pairs
percent = round(100 * unseen_pairs / total_pairs, 4)

print("Words used:", len(word_list))
print("Possible word pairs:", total_pairs)
print("Pairs seen in training:", seen_pairs)
print("Pairs never seen:", unseen_pairs)
print("Percentage never seen:", percent, "%")

Words used: 10
Possible word pairs: 100
Pairs seen in training: 13
Pairs never seen: 87
Percentage never seen: 87.0 %


#### What the output shows

Out of 100 possible word pairs only 13 are seen in training, so 87 pairs never occur. That is 87.0 percent. Even in a tiny corpus like this one, most of the possible pairs are missing, which is why the zero probability problem is so common.

### Part 4: Write safe_log

Here I write safe_log(p). The normal math.log(0) raises a math error. My function returns negative infinity when p is 0 and the normal ln otherwise. I use it to add the ln probabilities of all bigrams of the second test sentence.

In [25]:
def safe_log(p):
    if p == 0:
        return float("-inf")
    return math.log(p)

tokens = test_tokens[1]
total_log = 0
for i in range(1, len(tokens)):
    p = bigram_prob(tokens[i - 1], tokens[i])
    print(f"  P({tokens[i]} | {tokens[i - 1]}) = {round(p, 4)} | ln = {safe_log(p)}")
    total_log = total_log + safe_log(p)

print()
print("Log probability of the second test sentence:", total_log)

  P(the | <s>) = 1.0 | ln = 0.0
  P(fish | the) = 0.125 | ln = -2.0794415416798357
  P(sat | fish) = 0.0 | ln = -inf
  P(on | sat) = 1.0 | ln = 0.0
  P(the | on) = 1.0 | ln = 0.0
  P(mat | the) = 0.125 | ln = -2.0794415416798357
  P(</s> | mat) = 1.0 | ln = 0.0

Log probability of the second test sentence: -inf


#### What the output shows

Every bigram of the second sentence has a normal ln value, for example ln of 0.125 is about negative 2.0794, except P(sat | fish), which gives negative infinity. When I add negative infinity to the other numbers the result stays negative infinity, so the log probability of the whole sentence is negative infinity.

## Task 8: Smoothing

Smoothing fixes the zero probability problem. It takes a small part of the probability from the seen N grams and gives it to the unseen ones, so nothing is exactly 0.

### Part 1 and 2: Add one smoothing

Here I write bigram_prob_addone(prev, w) with the formula (C(v, w) + 1) / (C(v) + V). I add 1 to every count, and I add V to the bottom so that the probabilities still add up to 1. Then I print P(sat | fish) and P(cat | the) for MLE and for add one side by side.

In [26]:
def bigram_prob_addone(prev, w):
    return (bigram_counts.get((prev, w), 0) + 1) / (unigram_counts.get(prev, 0) + V)

print(f"{'':<16} {'MLE':<10} {'Add one'}")
for prev, w in [("fish", "sat"), ("the", "cat")]:
    mle = round(bigram_prob(prev, w), 4)
    add1 = round(bigram_prob_addone(prev, w), 4)
    print(f"{'P(' + w + ' | ' + prev + ')':<16} {mle:<10} {add1}")

                 MLE        Add one
P(sat | fish)    0.0        0.0833
P(cat | the)     0.25       0.1579


#### What the output shows

P(sat | fish) was 0.0 and is now 0.0833 (this is 1 / 12), so the unseen pair has a small chance. P(cat | the) goes down from 0.25 to 0.1579 (this is 3 / 19). The seen pair lost some probability and the unseen pair gained it.

### Part 3: Check that the probabilities sum to 1

Here I loop over every possible previous word and add the smoothed probabilities of all V words that can follow it. I use assert to check that the total is 1. I leave </s> out of the previous words because nothing ever comes after it, so it is never used as a previous word. I do include <s>.

In [27]:
previous_words = ["<s>"] + [w for w in vocab if w != "</s>"]

for prev in previous_words:
    total = 0
    for w in vocab:
        total += bigram_prob_addone(prev, w)
    print(f"{prev:<5} sum = {round(total, 4)}")
    assert abs(total - 1) < 1e-9

print()
print("All sums equal 1")

<s>   sum = 1.0
ate   sum = 1.0
bone  sum = 1.0
cat   sum = 1.0
dog   sum = 1.0
fish  sum = 1.0
mat   sum = 1.0
on    sum = 1.0
rug   sum = 1.0
sat   sum = 1.0
the   sum = 1.0

All sums equal 1


#### What the output shows

Every previous word has a sum of 1.0 and all the asserts passed. So add one smoothing gives a correct probability distribution for each previous word.

### Part 4: Add k smoothing

Here I write bigram_prob_addk(prev, w, k) with the formula (C(v, w) + k) / (C(v) + k * V). It is the same idea as add one, but I can choose how much I add. I print P(sat | fish) for k = 1, 0.5, 0.1 and 0.01 in a table.

In [28]:
def bigram_prob_addk(prev, w, k):
    return (bigram_counts.get((prev, w), 0) + k) / (unigram_counts.get(prev, 0) + k * V)

print(f"{'k':<6} {'P(sat | fish)'}")
for k in [1, 0.5, 0.1, 0.01]:
    print(f"{k:<6} {round(bigram_prob_addk('fish', 'sat', k), 4)}")

k      P(sat | fish)
1      0.0833
0.5    0.0769
0.1    0.0476
0.01   0.009


#### What the output shows

The probability of the unseen pair gets smaller when k gets smaller. It is 0.0833 for k = 1, 0.0769 for k = 0.5, 0.0476 for k = 0.1 and 0.009 for k = 0.01. A small k gives only a little probability to unseen pairs, and k = 1 gives the most. For k = 1 the result is the same as add one, as it should be.

## Task 9: Perplexity and model comparison

Perplexity tells how surprised a model is by the test data. A lower perplexity means a better model. It is the exponent of the negative average ln probability per token. In this task I use it to compare my models.

### Part 1 and 2: Perplexity of three models

Here I write perplexity(sentences, prob_fn). For each sentence it gets the probability of every predicted token (the words and </s>, but not <s>) from prob_fn, and adds up their ln values. If any probability is 0, it returns infinity. I compute the perplexity of the two test sentences together, which is 14 tokens, for the bigram MLE model, the bigram add one model and the unigram add one model, where P(w) = (C(w) + 1) / (N + V).

In [29]:
def perplexity(sentences, prob_fn):
    total_log = 0
    total_tokens = 0
    for sentence in sentences:
        tokens = tokenize(sentence)
        for p in prob_fn(tokens):
            if p == 0:
                return float("inf")
            total_log += math.log(p)
            total_tokens += 1
    return math.exp(-total_log / total_tokens)

def bigram_mle_probs(tokens):
    return [bigram_prob(tokens[i - 1], tokens[i]) for i in range(1, len(tokens))]

def bigram_addone_probs(tokens):
    return [bigram_prob_addone(tokens[i - 1], tokens[i]) for i in range(1, len(tokens))]

def unigram_addone_probs(tokens):
    return [(unigram_counts.get(w, 0) + 1) / (N + V) for w in tokens[1:]]

results = [
    ("Bigram MLE", perplexity(test, bigram_mle_probs)),
    ("Bigram add one", perplexity(test, bigram_addone_probs)),
    ("Unigram add one", perplexity(test, unigram_addone_probs)),
]

print(f"{'Model':<18} {'Perplexity'}")
for name, ppl in results:
    print(f"{name:<18} {round(ppl, 4)}")

Model              Perplexity
Bigram MLE         inf
Bigram add one     5.7969
Unigram add one    9.137


#### What the output shows

The bigram MLE model has infinite perplexity because of the one zero probability. The bigram add one model has a perplexity of 5.7969 and the unigram add one model has 9.137. The bigram model is better than the unigram model because it uses the previous word, and smoothing is what makes it work at all.

### Part 3: Add k bigram model for different k

Here I compute the perplexity of the add k bigram model for k = 1, 0.5, 0.1 and 0.01. Then I print the table and pick the k with the lowest perplexity.

In [30]:
k_values = [1, 0.5, 0.1, 0.01]
addk_results = {}

for k in k_values:
    def addk_probs(tokens):
        return [bigram_prob_addk(tokens[i - 1], tokens[i], k) for i in range(1, len(tokens))]
    addk_results[k] = perplexity(test, addk_probs)

print(f"{'k':<6} {'Perplexity'}")
for k in k_values:
    print(f"{k:<6} {round(addk_results[k], 4)}")

best_k = min(addk_results, key=addk_results.get)
print()
print("Best k:", best_k, "with perplexity", round(addk_results[best_k], 4))

k      Perplexity
1      5.7969
0.5    4.5703
0.1    2.9559
0.01   2.6272

Best k: 0.01 with perplexity 2.6272


#### What the output shows

The perplexity goes down when k goes down: 5.7969 for k = 1, 4.5703 for k = 0.5, 2.9559 for k = 0.1 and 2.6272 for k = 0.01. The best k on the test set is 0.01. The value for k = 1 is the same as the add one model above, so the two parts agree. Small k works well here because most test bigrams were seen in training and only one was not, so we should not give too much probability to the unseen ones.

### Part 4: Choose k on validation data

Here I train only on the first three training sentences, but I still build V from all four sentences, so V stays 11. I use the fourth sentence as validation data to choose k. Then I use that k on the test sentences. Finally I also look for the best k directly on the test set and compare the two results. In the table below the Validation column is the perplexity on the fourth sentence and the Test column is the perplexity on the two test sentences.

In [31]:
small_train = train[:3]
validation = train[3:]

small_unigrams = Counter()
small_bigrams = Counter()
for s in small_train:
    t = tokenize(s)
    small_unigrams.update(t)
    small_bigrams.update(get_ngrams(t, 2))

def small_addk_prob(prev, w, k):
    return (small_bigrams.get((prev, w), 0) + k) / (small_unigrams.get(prev, 0) + k * V)

def perplexity_small(sentences, k):
    def probs(tokens):
        return [small_addk_prob(tokens[i - 1], tokens[i], k) for i in range(1, len(tokens))]
    return perplexity(sentences, probs)

print("V (built from all four sentences):", V)
print()
print(f"{'k':<6} {'Validation':<14} {'Test'}")
val_results = {}
test_results = {}
for k in k_values:
    val_results[k] = perplexity_small(validation, k)
    test_results[k] = perplexity_small(test, k)
    print(f"{k:<6} {round(val_results[k], 4):<14} {round(test_results[k], 4)}")

best_val_k = min(val_results, key=val_results.get)
best_test_k = min(test_results, key=test_results.get)

print()
print("Best k on validation:", best_val_k, "| test perplexity with it:", round(test_results[best_val_k], 4))
print("Best k on test set  :", best_test_k, "| test perplexity with it:", round(test_results[best_test_k], 4))

V (built from all four sentences): 11

k      Validation     Test
1      8.5857         5.7406
0.5    8.0163         4.4539
0.1    8.026          2.7778
0.01   13.1304        2.4258

Best k on validation: 0.5 | test perplexity with it: 4.4539
Best k on test set  : 0.01 | test perplexity with it: 2.4258


#### What the output shows

On the validation sentence the best k is 0.5 with perplexity 8.0163 (k = 0.1 is very close with 8.026). The reason is that three of the six bigrams of the validation sentence are unseen, so this sentence wants a larger k. With k = 0.5 the test perplexity is 4.4539. When I choose k directly on the test set, the best k is 0.01 and the perplexity is 2.4258.

So the k chosen on validation is different and gives a higher test perplexity. The 4.4539 is the honest number, because the test data was not used to pick k. The 2.4258 is too optimistic, because I tuned k on the same data I measure. Also, my validation set is only one sentence, so it is a weak guide, and with more validation data the chosen k would be more reliable.